# VectorTrainer Benchmark

Compare `SyncVectorEnv` and `AsyncVectorEnv` inside `VectorTrainer.train(...)`. This benchmark is intentionally below HPO orchestration: no `StudyRunner`, no dashboard, no Optuna, no checkpointing, and no evaluation.

## Setup

In [ ]:
# cell: setup

from pathlib import Path
import os
import platform
import subprocess
import sys
import time

if not Path("hpo/src").exists():
    if not Path("/content").exists():
        raise RuntimeError("Run this notebook from the rl_lab repository root or in Colab.")
    subprocess.run(["git", "clone", "https://github.com/miwehle/rl_lab.git"], check=True)
    os.chdir("rl_lab")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "hpo/requirements.txt"],
    check=True,
)

for _path in ("dqn/src", "hpo/src"):
    if _path not in sys.path:
        sys.path.insert(0, _path)

import gymnasium as gym
import numpy as np
import pandas as pd
import torch
from gymnasium.vector import AsyncVectorEnv

from dqn.vector_training import VectorTrainer, VectorTrainingConfig
from hpo.solar_system_lander.environment import DEFAULT_WORLD_MIX, EnvFactory

def gpu_name():
    try:
        return subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            text=True,
        ).strip()
    except Exception:
        return "no nvidia-smi"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("python:", platform.python_version())
print("gymnasium:", gym.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", gpu_name())
print("device:", device)

## CPU Info

In [ ]:
# cell: cpu-info; requires: setup

CPU_COUNT = os.cpu_count() or 2
WORLD_COUNT = len(DEFAULT_WORLD_MIX)

try:
    import psutil
except ImportError:
    psutil = None

print("logical cpu_count:", CPU_COUNT)

if psutil is not None:
    print("physical cores:", psutil.cpu_count(logical=False))
    print("logical cores:", psutil.cpu_count(logical=True))

def next_multiple(value, divisor):
    return ((value + divisor - 1) // divisor) * divisor

NUM_ENVS_LIST = sorted({
    next_multiple(CPU_COUNT, WORLD_COUNT),
    next_multiple(2 * CPU_COUNT, WORLD_COUNT),
    next_multiple(4 * CPU_COUNT, WORLD_COUNT),
})

print("world_count:", WORLD_COUNT)
print("NUM_ENVS_LIST:", NUM_ENVS_LIST)

## Benchmark Config

In [ ]:
# cell: benchmark-config; requires: cpu-info

OBSERVATION_MODE = "10d"
WORLD_MIX = DEFAULT_WORLD_MIX
PARAMS = {}

SEED = 123
REPEATS = 1

# Keep this short: it is a trainer microbenchmark, not a model-quality run.
TRAINING_CFG = VectorTrainingConfig(
    num_episodes=200,
    batch_size=512,
    eps_start=1.0,
    eps_end=0.04400876385215351,
    eps_decay=38_793,
    learning_rate=0.0006229370728793535,
    gamma=0.995,
    tau=0.002,
    learning_starts=2_500,
    optimize_every=2,
    hidden_size=128,
    adaptive_extension_window=None,
    early_stopping_score=None,
)
REPLAY_MEMORY_CAPACITY = 76_754

FACTORY = EnvFactory(OBSERVATION_MODE, world_mix=WORLD_MIX)

print("worlds:", [world.name for world in FACTORY.worlds])
print("num_envs:", NUM_ENVS_LIST)
print("repeats:", REPEATS)
print("training episodes per run:", TRAINING_CFG.num_episodes)
print("batch_size:", TRAINING_CFG.batch_size)
print("learning_starts:", TRAINING_CFG.learning_starts)
print("optimize_every:", TRAINING_CFG.optimize_every)

## Benchmark Helpers

In [ ]:
# cell: benchmark-helpers; requires: benchmark-config

def training_fns(factory, num_envs, params):
    if num_envs % len(factory.worlds):
        raise ValueError(f"num_envs must be divisible by {len(factory.worlds)}")
    slots_per_world = num_envs // len(factory.worlds)
    # Experimental notebook code: reuse the current implementation factories without changing production API yet.
    return [
        factory._training_factory(world, params)
        for world in factory.worlds
        for _ in range(slots_per_world)
    ]

def make_vector_env(kind, num_envs):
    if kind == "sync":
        return FACTORY.make_training_env(num_envs, params=PARAMS)
    if kind == "async":
        return AsyncVectorEnv(training_fns(FACTORY, num_envs, PARAMS))
    raise ValueError(f"unknown vector env kind: {kind}")

def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def mean_last(values, n):
    if not values:
        return None
    tail = values[-min(n, len(values)) :]
    return float(np.mean(tail))

def peak_cuda_memory_mb():
    if not torch.cuda.is_available():
        return None
    return torch.cuda.max_memory_allocated() / 1024 / 1024

def measure_once(kind, num_envs, repeat):
    env = make_vector_env(kind, num_envs)
    try:
        trainer = VectorTrainer(
            env,
            seed=SEED + repeat,
            device=device,
            replay_memory_capacity=REPLAY_MEMORY_CAPACITY,
            hidden_size=TRAINING_CFG.hidden_size,
        )
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        cuda_sync()
        started = time.perf_counter()
        result = trainer.train(TRAINING_CFG)
        cuda_sync()
        seconds = time.perf_counter() - started
        episodes = len(result.episode_returns)
        return {
            "kind": kind,
            "num_envs": num_envs,
            "repeat": repeat,
            "status": "ok",
            "seconds": seconds,
            "episodes": episodes,
            "env_steps": result.env_steps,
            "optimizer_updates": result.optimizer_updates,
            "episodes_per_second": episodes / seconds,
            "env_steps_per_second": result.env_steps / seconds,
            "optimizer_updates_per_second": result.optimizer_updates / seconds,
            "mean_return": float(np.mean(result.episode_returns)),
            "last_50_mean_return": mean_last(result.episode_returns, 50),
            "peak_cuda_memory_mb": peak_cuda_memory_mb(),
            "error": "",
        }
    finally:
        env.close()

## Run Benchmark

In [ ]:
# cell: run-benchmark; requires: benchmark-helpers

rows = []

for num_envs in NUM_ENVS_LIST:
    for kind in ("sync", "async"):
        for repeat in range(REPEATS):
            try:
                row = measure_once(kind, num_envs, repeat)
                print(
                    f"{kind:5s} num_envs={num_envs:2d} repeat={repeat} "
                    f"{row['episodes_per_second']:7.2f} episodes/s "
                    f"{row['env_steps_per_second']:9.0f} env-steps/s "
                    f"{row['optimizer_updates_per_second']:7.1f} updates/s "
                    f"({row['seconds']:.2f}s)"
                )
            except Exception as exc:
                row = {
                    "kind": kind,
                    "num_envs": num_envs,
                    "repeat": repeat,
                    "status": "fail",
                    "seconds": None,
                    "episodes": None,
                    "env_steps": None,
                    "optimizer_updates": None,
                    "episodes_per_second": None,
                    "env_steps_per_second": None,
                    "optimizer_updates_per_second": None,
                    "mean_return": None,
                    "last_50_mean_return": None,
                    "peak_cuda_memory_mb": None,
                    "error": repr(exc),
                }
                print(f"{kind:5s} num_envs={num_envs:2d} repeat={repeat} FAIL: {row['error']}")
            rows.append(row)

## Results

In [ ]:
# cell: results; requires: run-benchmark

df = pd.DataFrame(rows)
display(df)

ok = df[df["status"] == "ok"].copy()
metrics = [
    "seconds",
    "episodes_per_second",
    "env_steps_per_second",
    "optimizer_updates_per_second",
    "peak_cuda_memory_mb",
]
summary = ok.groupby(["kind", "num_envs"], as_index=False)[metrics].mean()
display(summary)

pivot = summary.pivot(index="num_envs", columns="kind", values="seconds")
if {"sync", "async"}.issubset(pivot.columns):
    pivot["sync_seconds_over_async_seconds"] = pivot["sync"] / pivot["async"]
display(pivot)

throughput = summary.pivot(index="num_envs", columns="kind", values="env_steps_per_second")
if {"sync", "async"}.issubset(throughput.columns):
    throughput["async_vs_sync"] = throughput["async"] / throughput["sync"]
display(throughput)

## Reading the Result

This benchmark includes environment stepping, replay-buffer sampling, CPU-to-GPU batch transfer, forward passes, backward passes, and optimizer updates. It still excludes Optuna, StudyRunner, dashboard rendering, checkpointing, and greedy evaluation. If `AsyncVectorEnv` improves this notebook materially, the next step is a small production API switch; if not, the env-step speedup is being eaten by training-side costs.